# 11. Interaktív Ingatlan Kereső és Portfólió Dashboard

**Cél**: Felhasználóbarát, dinamikus döntéstámogató felület a kőbányai lakásállomány szűrésére és elemzésére.

---

### 📖 Mire használható a dashboard?
A teljes 1 320 darabos adatbázisban böngészhet ár, alapterület, szobaszám, városrész és típus szerint, azonnali élő statisztikákkal.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from _utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np

df = load_szamitott_master()
print(f"Teljes adatbázis betöltve: {len(df)} hirdetés.")

Teljes adatbázis betöltve: 1320 hirdetés.


### 1. Dinamikus Szűrők és Reagáló KPI Kártyák

**📌 Mit lát a vezérlőpult tetején?**
A szűrés után azonnal frissül a találatok száma, az átlagár, a medián négyzetméterár és az átlagos méret.

In [2]:
n_all = len(df)
n_elado = len(df[df['listing_type'] == 'elado'])
n_kiado = len(df[df['listing_type'] == 'kiado'])
med_ar_elado = df[df['listing_type'] == 'elado']['price_huf'].median()
med_nm_elado = df[df['listing_type'] == 'elado']['nm_ar_huf'].median()
atlag_meret = df['alapterulet_nm'].mean()

kpi_cards = [
    ("Összes Kínálat", f"{n_all:,} db".replace(',', ' '), "Teljes adatbázis", "#1e3a8a"),
    ("Eladó Lakások", f"{n_elado:,} db".replace(',', ' '), f"Medián: {fmt_mft(med_ar_elado/1e6)}", "#2563eb"),
    ("Kiadó Lakások", f"{n_kiado:,} db".replace(',', ' '), f"{n_kiado/n_all*100:.1f}% arány", "#d97706"),
    ("Eladó Medián Ár/m²", fmt_huf(med_nm_elado), "Fajlagos ár", "#10b981"),
    ("Átlagos Méret", f"{atlag_meret:.1f} m²", "Kínálati átlag", "#059669"),
    ("Városrészek", f"{df['varosresz'].nunique()} zóna", "Kőbánya egésze", "#7c3aed")
]
display(HTML(kpi_grid_html(kpi_cards)))

### 2. Térképi Elhelyezkedés és Találati Lista

**📌 Hogyan használható a találati lista?**
A mini térképen láthatók a szűrt lakások, a táblázat pedig rendezhető ár, m² ár vagy méret szerint.

In [3]:
fig1 = px.histogram(
    df,
    x='nm_ar_huf',
    color='listing_type',
    barmode='overlay',
    nbins=40,
    title='Kínálati Négyzetméterárak Eloszlása Eladó és Kiadó Szegmensekben',
    labels={'nm_ar_huf': 'Ár / m² (HUF)', 'listing_type': 'Típus'},
    template=PLOTLY_TEMPLATE,
    opacity=0.75
)
fig1.update_layout(height=420)
fig1.show()

# Pontos minták térképe
df_pts = df[(df['minta_garantalt_pontos'] == 1) & (df['listing_type'] == 'elado')]
fig2 = px.scatter_map(
    df_pts,
    lat='geokodolt_lat',
    lon='geokodolt_lon',
    color='nm_ar_huf',
    size='alapterulet_nm',
    hover_name='cim_teljes',
    hover_data={'nm_ar_huf': ':.0f', 'varosresz': True},
    zoom=12.2,
    center={'lat': KOBANYA_CENTER_LAT, 'lon': KOBANYA_CENTER_LON},
    map_style='carto-positron',
    title=f'Garantált pontos eladó lakások térképi elhelyezkedése (N={len(df_pts)})'
)
fig2.update_layout(height=480, margin={"r":0,"t":40,"l":0,"b":0})
fig2.show()

### 3. Áreloszlási Hisztogram

**📌 Mit mutat a kisegítő ábra?**
A kiválasztott részpiac belső árszóródását és homogenitását ábrázolja.

In [4]:
show_cols = ['listing_id', 'cim_teljes', 'ar_millio_ft', 'nm_ar_huf', 'alapterulet_nm', 'szobaszam_osszes', 'varosresz', 'allapot']
top_listings = df[df['listing_type'] == 'elado'][show_cols].head(25).copy()

top_listings['ar_millio_ft'] = top_listings['ar_millio_ft'].apply(lambda x: f"{x:.1f} M Ft" if pd.notna(x) else "—")
top_listings['nm_ar_huf'] = top_listings['nm_ar_huf'].apply(fmt_huf)
top_listings['alapterulet_nm'] = top_listings['alapterulet_nm'].apply(lambda x: f"{x:.0f} m²" if pd.notna(x) else "—")

table_html = "<div style='overflow-x:auto; max-height:450px; overflow-y:auto; margin:15px 0;'>" + top_listings.to_html(classes='table table-striped table-hover', index=False) + "</div>"
display(HTML(table_html))

listing_id,cim_teljes,ar_millio_ft,nm_ar_huf,alapterulet_nm,szobaszam_osszes,varosresz,allapot
32387225,"Budapest X. kerület, Szövőszék utca",54.9 M Ft,1 098 000 Ft,50 m²,2.0,Újhegy,felújítandó
33203915,"Budapest X. kerület, Nyerő utca 7.",67.9 M Ft,1 060 938 Ft,64 m²,3.0,Laposdűlő,jó állapotú
33365461,"Budapest X. kerület, Maláta köz",119.0 M Ft,1 252 632 Ft,95 m²,4.0,Óhegy,újszerű
33434706,"Budapest X. kerület, Hungária körút",89.0 M Ft,1 534 483 Ft,58 m²,3.0,Laposdűlő,felújított
33805283,"Budapest X. kerület, Tárna utca",23.8 M Ft,849 643 Ft,28 m²,1.0,Gyárdűlő,felújított
33939524,"Budapest X. kerület, Hős utca 15/A.",35.9 M Ft,1 436 000 Ft,25 m²,1.0,Laposdűlő,felújítandó
34084830,"Budapest X. kerület, Tavas utca 7.",102.2 M Ft,1 648 387 Ft,62 m²,3.0,Újhegy,új építésű
34085875,"Budapest X. kerület, Tavas utca 7.",119.9 M Ft,1 598 667 Ft,75 m²,3.0,Újhegy,új építésű
34103639,"Budapest X. kerület, Kőbánya - Óhegy",118.4 M Ft,1 195 758 Ft,99 m²,4.0,Óhegy,nincs megadva
34159451,"Budapest X. kerület, Dömsödi utca",62.0 M Ft,1 215 686 Ft,51 m²,1.0,Felsőrákos,jó állapotú


### 4. Döntéstámogató Alkalmazás

**📌 Gyakorlati haszon**:
Közvetlenül alkalmas piaci összehasonlító elemzések és vásárlási döntések előkészítésére.

In [5]:
w_tipus = widgets.RadioButtons(options=['eladó', 'kiadó', 'Mindkettő'], value='eladó', description='Típus:')
w_ar_sav = widgets.IntRangeSlider(min=20, max=150, value=(30, 80), description='Ár (M Ft):')
w_meret_sav = widgets.IntRangeSlider(min=20, max=150, value=(35, 90), description='Méret (m²):')
w_varos = widgets.SelectMultiple(options=VAROSRESZEK, value=tuple(VAROSRESZEK[:3]), description='Városrészek:')
w_rendezes = widgets.Dropdown(options=['Ár növekvő', 'Ár csökkenő', 'Ár/m² növekvő', 'Méret csökkenő'], value='Ár/m² növekvő', description='Rendezés:')

out_search = widgets.Output()

def frissit_kereses(*args):
    dff = df.copy()
    if w_tipus.value != 'Mindkettő':
        mapped_val = 'elado' if w_tipus.value == 'eladó' else 'kiado'
        dff = dff[dff['listing_type'] == mapped_val]
    if w_tipus.value == 'eladó':
        dff = dff[(dff['ar_millio_ft'] >= w_ar_sav.value[0]) & (dff['ar_millio_ft'] <= w_ar_sav.value[1])]
    dff = dff[(dff['alapterulet_nm'] >= w_meret_sav.value[0]) & (dff['alapterulet_nm'] <= w_meret_sav.value[1])]
    if w_varos.value:
        dff = dff[dff['varosresz'].isin(w_varos.value)]
        
    if w_rendezes.value == 'Ár növekvő':
        dff = dff.sort_values('price_huf', ascending=True)
    elif w_rendezes.value == 'Ár csökkenő':
        dff = dff.sort_values('price_huf', ascending=False)
    elif w_rendezes.value == 'Ár/m² növekvő':
        dff = dff.sort_values('nm_ar_huf', ascending=True)
    else:
        dff = dff.sort_values('alapterulet_nm', ascending=False)
        
    with out_search:
        clear_output(wait=True)
        search_kpis = [
            ("Találatok Száma", f"{len(dff):,} db".replace(',', ' '), "Szűrési feltételeknek megfelel", "#1e3a8a"),
            ("Szűrt Medián Ár", fmt_mft(dff['price_huf'].median()/1e6 if len(dff)>0 else 0), "Középérték", "#2563eb"),
            ("Szűrt Medián Ár/m²", fmt_huf(dff['nm_ar_huf'].median() if len(dff)>0 else 0), "Fajlagos", "#10b981")
        ]
        display(HTML(kpi_grid_html(search_kpis)))
        if len(dff) > 0:
            preview = dff[['cim_teljes', 'ar_millio_ft', 'nm_ar_huf', 'alapterulet_nm', 'szobaszam_osszes', 'varosresz']].head(20).copy()
            preview['ar_millio_ft'] = preview['ar_millio_ft'].apply(lambda x: f"{x:.1f} M" if pd.notna(x) else "—")
            preview['nm_ar_huf'] = preview['nm_ar_huf'].apply(fmt_huf)
            display(HTML("<div style='overflow-x:auto; max-height:350px; overflow-y:auto;'>" + preview.to_html(classes='table table-sm table-striped', index=False) + "</div>"))
        else:
            print("Nincs a feltételeknek megfelelő találat.")

w_tipus.observe(frissit_kereses, names='value')
w_ar_sav.observe(frissit_kereses, names='value')
w_meret_sav.observe(frissit_kereses, names='value')
w_varos.observe(frissit_kereses, names='value')
w_rendezes.observe(frissit_kereses, names='value')

display(widgets.VBox([
    widgets.HBox([w_tipus, w_ar_sav, w_meret_sav]),
    widgets.HBox([w_varos, w_rendezes])
]))
display(out_search)
frissit_kereses()

Output()